
# HGT Top-k Recommendation Notebook

This notebook redesigns the recommendation pipeline around your **final training Option B production model**.

Core design:

1. Load `hetero_data15.pt`
2. Load the fold checkpoints from `hgt_final_optionB_outputs/`.
3. Defaults to a **5-fold ensemble average score**, which makes recommendations more stable than a single fold.
4. Excludes existing `collaborate_with` collaboration edges, to avoid recommending channels already collaborated with.
5. Provides three scores:
   - `raw_prob`: the model's raw collaboration probability score.
   - `baseline_corrected`: subtracts a channel's average score against random brands, to reduce the influence of generically high-scoring channels.
   - `final_score`: adds a channel degree penalty on top of the baseline-corrected score, to reduce popularity bias toward popular channels.
6. Ranks directly by `final_score`; MMR re-ranking is no longer used.

> How to run: execute all cells in order. Just edit `BRAND_LIST` and the paths in the last cell.


In [ ]:

import os, json, glob, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HGTConv
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

# Basic configuration: aligned with final training
SEED = 42
DATA_PATH = 'YOUR_GRAPH_OUTPUT_FILE.pt'
JSON_PATH = 'YOUR_DATASET_FILE.json'
CHECKPOINT_DIR = 'hgt_final_optionB_outputs'
CHECKPOINT_GLOB = 'hgt_final_optionB_fold*_best.pt'

# Final Option B params
OPTION_B_PARAMS = dict(
    hidden_dim=128,
    out_dim=128,
    num_heads=4,
    num_layers=2,
    dropout=0.5241,
    lr=0.001009,
    weight_decay=0.000567,
)

# Target edge
TARGET_EDGE_TYPE = ('youtuber', 'collaborate_with', 'brand')
YOUTUBER_NODE_TYPE = 'youtuber'
BRAND_NODE_TYPE = 'brand'

# Default recommendation settings
SCORING_MODE = 'ensemble'     
SCORE_TYPE = 'final_score'    
NUM_BASELINE_BRANDS = 80      
POPULARITY_ALPHA = 0.08       
CHUNK_SIZE = 2048             
TOP_K = 10

if torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_MPS_IF_AVAILABLE and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('Device:', device)


## 1. Final model architecture

In [ ]:

class HGTEncoder(nn.Module):
    def __init__(self, metadata, in_dims, hidden_dim, out_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.dropout = dropout
        self.input_lin = nn.ModuleDict({nt: nn.Linear(d, hidden_dim) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HGTConv(hidden_dim, hidden_dim, metadata=metadata, heads=num_heads)
            for _ in range(num_layers)
        ])
        self.output_lin = nn.ModuleDict({nt: nn.Linear(hidden_dim, out_dim) for nt in metadata[0]})

    def forward(self, x_dict, edge_index_dict):
        x_dict = {nt: F.relu(self.input_lin[nt](x.float())) for nt, x in x_dict.items()}
        for conv in self.convs:
            res = {nt: x.clone() for nt, x in x_dict.items()}
            xout = conv(x_dict, edge_index_dict)
            x_dict = {
                nt: F.dropout(F.relu(xout[nt] + res[nt]), p=self.dropout, training=self.training)
                for nt in xout
            }
        return {nt: self.output_lin[nt](x) for nt, x in x_dict.items()}


class MLPDecoder(nn.Module):
    def __init__(self, in_dim, dropout):
        super().__init__()
        h = in_dim
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 4, h), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h // 2, 1),
        )

    def forward(self, z_src, z_dst, eli):
        s, d = eli
        sv, dv = z_src[s], z_dst[d]
        feat = torch.cat([sv, dv, sv * dv, (sv - dv).abs()], dim=-1)
        return self.mlp(feat).squeeze(-1)


class HGTModel(nn.Module):
    def __init__(self, metadata, in_dims, hidden_dim, out_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.encoder = HGTEncoder(metadata, in_dims, hidden_dim, out_dim, num_heads, num_layers, dropout)
        self.decoder = MLPDecoder(out_dim, dropout)

    def forward(self, hdata, tgt):
        z = self.encoder(hdata.x_dict, hdata.edge_index_dict)
        src_t, _, dst_t = tgt
        eli = hdata[tgt].edge_label_index
        return self.decoder(z[src_t], z[dst_t], eli), z

print('Model classes loaded.')


## 2. Load graph data and build lookup tables

In [ ]:
def load_graph(data_path=DATA_PATH):
    print(f'Loading graph: {data_path}')
    data = torch.load(data_path, weights_only=False)
    data = apply_final_pca_whitening(data)
    print('Node types:', data.node_types)
    print('Edge types:', data.edge_types)
    print('Feature dims:', {nt: data[nt].x.size(-1) for nt in data.node_types})
    return data


def _safe_get_summary(row):
    if isinstance(row, dict):
        return row.get('summary', row)
    return {}


def build_lookup_from_dataset_json(json_path, data):
    """
    Build the brand index and youtuber/channel index lookup from the dataset json.
    NOTE: this uses a sorted set, which must match the mapping rule used when
    building the graph. If you have the original mapping file, consider using
    load_lookup_from_mapping_files() instead.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    # brand mapping
    all_brands = sorted(set(
        b.strip()
        for d in raw
        for b in (_safe_get_summary(d).get('brand名稱_標準化') or '').split(',')
        if b.strip()
    ))
    brand_to_idx = {name: idx for idx, name in enumerate(all_brands)}
    idx_to_brand = {idx: name for name, idx in brand_to_idx.items()}

    # youtuber/channel mapping
    all_channel_ids = sorted(set(
        _safe_get_summary(d).get('頻道ID')
        for d in raw
        if _safe_get_summary(d).get('頻道ID')
    ))

    ch_id_to_name = {}
    for d in raw:
        s = _safe_get_summary(d)
        cid = s.get('頻道ID')
        name = s.get('頻道名稱')
        if cid and name and cid not in ch_id_to_name:
            ch_id_to_name[cid] = name

    idx_to_youtuber = {
        idx: ch_id_to_name.get(cid, cid)
        for idx, cid in enumerate(all_channel_ids)
    }
    youtuber_to_idx = {name: idx for idx, name in idx_to_youtuber.items()}

    # basic warnings
    n_brand_graph = data[BRAND_NODE_TYPE].num_nodes
    n_y_graph = data[YOUTUBER_NODE_TYPE].num_nodes
    if len(brand_to_idx) != n_brand_graph:
        print(f'brand mapping count ({len(brand_to_idx)}) != graph brand nodes ({n_brand_graph}). Please check whether the json matches the hetero_data15.pt version.')
    if len(idx_to_youtuber) != n_y_graph:
        print(f'youtuber mapping count ({len(idx_to_youtuber)}) != graph youtuber nodes ({n_y_graph}). Please check whether the json matches the hetero_data15.pt version.')

    return brand_to_idx, idx_to_brand, youtuber_to_idx, idx_to_youtuber


def search_brand(keyword, brand_to_idx, top_n=30):
    keyword = keyword.lower()
    matches = [(name, idx) for name, idx in brand_to_idx.items() if keyword in name.lower()]
    print(f'Found {len(matches)} matches for "{keyword}"')
    for name, idx in matches[:top_n]:
        print(f'  {idx:>5}  {name}')
    return matches


data = load_graph(DATA_PATH)
brand_to_idx, idx_to_brand, youtuber_to_idx, idx_to_youtuber = build_lookup_from_dataset_json(JSON_PATH, data)
print(f'Lookup loaded: brands={len(brand_to_idx)}, youtubers={len(idx_to_youtuber)}')


## 3. Load final checkpoints

In [ ]:
def find_checkpoint_paths(checkpoint_dir=CHECKPOINT_DIR, pattern=CHECKPOINT_GLOB):
    paths = sorted(glob.glob(os.path.join(checkpoint_dir, pattern)))
    if not paths:
        raise FileNotFoundError(
            f"Checkpoint not found: {os.path.join(checkpoint_dir, pattern)}"
            " Please confirm you have run hgt_final_training_optionB_formal.ipynb and that the checkpoint directory path is correct."
        )
    return paths


def load_model_from_checkpoint(path, data, device=device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    params = ckpt.get('params', OPTION_B_PARAMS)
    in_dims = ckpt.get('in_dims', {nt: data[nt].x.size(-1) for nt in data.node_types})
    metadata = data.metadata()

    # Check whether the feature dims are consistent
    current_dims = {nt: data[nt].x.size(-1) for nt in data.node_types}
    if in_dims != current_dims:
        print('checkpoint in_dims != current data dims')
        print('checkpoint:', in_dims)
        print('current   :', current_dims)
        print('This usually means PCA whitening settings or the data file version do not match.')

    model = HGTModel(
        metadata=metadata,
        in_dims=in_dims,
        hidden_dim=params['hidden_dim'],
        out_dim=params['out_dim'],
        num_heads=params['num_heads'],
        num_layers=params['num_layers'],
        dropout=params['dropout'],
    ).to(device)

    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    model.eval()
    return model, ckpt


def load_final_models(scoring_mode=SCORING_MODE):
    paths = find_checkpoint_paths()
    ckpts = []

    if scoring_mode == 'best_fold':
        # Use the checkpoint's test_auc as the selection criterion for a single fold
        meta = []
        for p in paths:
            c = torch.load(p, map_location='cpu', weights_only=False)
            auc = c.get('metrics', {}).get('test_auc', -1)
            meta.append((auc, p))
        paths = [max(meta)[1]]
        print(f'Using best fold checkpoint only: {paths[0]}')
    else:
        print(f'Using ensemble checkpoints: {len(paths)} folds')

    models = []
    checkpoint_infos = []
    for p in paths:
        model, ckpt = load_model_from_checkpoint(p, data, device)
        models.append(model)
        checkpoint_infos.append({
            'path': p,
            'fold': ckpt.get('fold'),
            **ckpt.get('metrics', {})
        })

    info_df = pd.DataFrame(checkpoint_infos)
    display(info_df[[c for c in ['fold', 'test_auc', 'test_ap', 'best_epoch', 'eff_rank', 'mean_cos', 'path'] if c in info_df.columns]])
    return models, info_df

models, checkpoint_info = load_final_models(SCORING_MODE)

## 4. Compute embeddings

In [ ]:

@torch.no_grad()
def get_embeddings_for_models(models, data, device=device):
    data = data.to(device)
    all_z = []
    for i, model in enumerate(models, start=1):
        model.eval()
        z = model.encoder(data.x_dict, data.edge_index_dict)
        all_z.append({
            YOUTUBER_NODE_TYPE: z[YOUTUBER_NODE_TYPE].detach(),
            BRAND_NODE_TYPE: z[BRAND_NODE_TYPE].detach(),
        })
        print(f'Embeddings ready: model {i}/{len(models)}')
    return all_z

all_embeddings = get_embeddings_for_models(models, data, device)

existing_collab = set(zip(
    data[TARGET_EDGE_TYPE].edge_index[0].detach().cpu().tolist(),
    data[TARGET_EDGE_TYPE].edge_index[1].detach().cpu().tolist(),
))
print(f'Existing collaborate_with edges: {len(existing_collab)}')

# Channel degree: used for popularity correction
num_y = data[YOUTUBER_NODE_TYPE].num_nodes
channel_degree = torch.zeros(num_y, dtype=torch.float)
for src, dst in existing_collab:
    channel_degree[src] += 1
channel_degree_norm = torch.log1p(channel_degree)
if channel_degree_norm.max() > 0:
    channel_degree_norm = channel_degree_norm / channel_degree_norm.max()
print('Channel degree summary:')
print(pd.Series(channel_degree.numpy()).describe())


## 5. Scoring functions

In [ ]:

def resolve_brand(brand_query, brand_to_idx, exact=False, show_matches=True):
    """Brand name lookup: supports index, full name, or keyword."""
    if isinstance(brand_query, int):
        return brand_query, idx_to_brand.get(brand_query, f'brand_idx={brand_query}')

    q = str(brand_query).strip()
    if q in brand_to_idx:
        return brand_to_idx[q], q

    q_lower = q.lower()
    if exact:
        matches = [(name, idx) for name, idx in brand_to_idx.items() if name.lower() == q_lower]
    else:
        matches = [(name, idx) for name, idx in brand_to_idx.items() if q_lower in name.lower()]

    if not matches:
        raise ValueError(f'Brand not found: {brand_query}. Try search_brand("keyword", brand_to_idx) first.')

    if show_matches and len(matches) > 1:
        print(f'Found multiple brands matching "{brand_query}", using the first one by default:')
        for name, idx in matches[:20]:
            print(f'  {idx:>5}  {name}')
        if len(matches) > 20:
            print('  ...')

    brand_name, brand_idx = matches[0]
    return brand_idx, brand_name


@torch.no_grad()
def _score_one_model(model, z_y, z_b, brand_idx, candidates, baseline_brand_ids, chunk_size=CHUNK_SIZE):
    """Return a single model's raw_prob and baseline_corrected for all candidates."""
    device = z_y.device
    raw_parts = []
    base_parts = []

    for start in range(0, len(candidates), chunk_size):
        chunk = candidates[start:start + chunk_size]
        cand_t = torch.tensor(chunk, dtype=torch.long, device=device)
        brand_t = torch.full((len(chunk),), int(brand_idx), dtype=torch.long, device=device)
        eli = torch.stack([cand_t, brand_t], dim=0)
        raw_prob = torch.sigmoid(model.decoder(z_y, z_b, eli)).detach()

        # baseline: the average score of the same batch of channels against random brands
        base_sum = torch.zeros(len(chunk), device=device)
        for b in baseline_brand_ids:
            b_t = torch.full((len(chunk),), int(b), dtype=torch.long, device=device)
            eli_b = torch.stack([cand_t, b_t], dim=0)
            base_sum += torch.sigmoid(model.decoder(z_y, z_b, eli_b)).detach()
        baseline = base_sum / max(1, len(baseline_brand_ids))

        raw_parts.append(raw_prob.cpu())
        base_parts.append((raw_prob - baseline).cpu())

    return torch.cat(raw_parts), torch.cat(base_parts)


@torch.no_grad()
def score_candidates_ensemble(
    brand_idx,
    models,
    all_embeddings,
    existing_collab,
    num_baseline_brands=NUM_BASELINE_BRANDS,
    popularity_alpha=POPULARITY_ALPHA,
    seed=SEED,
):
    """
    Compute the scores of all non-collaborating youtubers for a given brand.

    final_score = mean(baseline_corrected across folds) - popularity_alpha * normalized_log_degree
    """
    already = {src for src, dst in existing_collab if dst == brand_idx}
    candidates = [i for i in range(data[YOUTUBER_NODE_TYPE].num_nodes) if i not in already]
    if not candidates:
        raise RuntimeError('This brand has no recommendable candidates.')

    rng = np.random.default_rng(seed + int(brand_idx))
    n_brand = data[BRAND_NODE_TYPE].num_nodes
    baseline_brand_ids = rng.choice(n_brand, size=min(num_baseline_brands, n_brand), replace=False).tolist()

    raw_list = []
    corr_list = []
    for model, z in zip(models, all_embeddings):
        z_y = z[YOUTUBER_NODE_TYPE]
        z_b = z[BRAND_NODE_TYPE]
        raw, corr = _score_one_model(model, z_y, z_b, brand_idx, candidates, baseline_brand_ids)
        raw_list.append(raw)
        corr_list.append(corr)

    raw_prob = torch.stack(raw_list, dim=0).mean(dim=0)
    baseline_corrected = torch.stack(corr_list, dim=0).mean(dim=0)
    ensemble_std = torch.stack(corr_list, dim=0).std(dim=0) if len(corr_list) > 1 else torch.zeros_like(baseline_corrected)

    deg_penalty = channel_degree_norm[candidates]
    final_score = baseline_corrected - popularity_alpha * deg_penalty

    scores = pd.DataFrame({
        'youtuber_idx': candidates,
        'channel_name': [idx_to_youtuber.get(i, f'idx={i}') for i in candidates],
        'raw_prob': raw_prob.numpy(),
        'baseline_corrected': baseline_corrected.numpy(),
        'degree_penalty': deg_penalty.numpy(),
        'ensemble_std': ensemble_std.numpy(),
        'final_score': final_score.numpy(),
        'existing_collab_degree': channel_degree[candidates].numpy(),
    })
    scores = scores.sort_values('final_score', ascending=False).reset_index(drop=True)
    scores['candidate_rank_pct'] = (np.arange(len(scores)) + 1) / len(scores) * 100
    return scores


## 6. Recommendation API

In [ ]:
def recommend_brand(
    brand_query,
    top_k=TOP_K,
    score_type=SCORE_TYPE,
    popularity_alpha=POPULARITY_ALPHA,
    export_csv=True,
    output_dir='recommendation_outputs',
):
    """Generate Top-k channel recommendations for a single brand."""
    os.makedirs(output_dir, exist_ok=True)

    brand_idx, brand_name = resolve_brand(brand_query, brand_to_idx, exact=False, show_matches=True)
    print('\n' + '=' * 90)
    print(f'Brand: {brand_name} | brand_idx={brand_idx}')
    print(f'Scoring mode: {SCORING_MODE} | score_type={score_type}')
    print('=' * 90)

    score_df = score_candidates_ensemble(
        brand_idx,
        models,
        all_embeddings,
        existing_collab,
        num_baseline_brands=NUM_BASELINE_BRANDS,
        popularity_alpha=popularity_alpha,
        seed=SEED,
    )
    rec = score_df.head(top_k).copy()
    rec.insert(0, 'rank', np.arange(1, len(rec) + 1))

    already_n = sum(1 for _, dst in existing_collab if dst == brand_idx)
    print(f'Excluded existing collaborated channels: {already_n}')
    print(f'Number of candidate channels: {len(score_df)}')


    cols = [
        'rank', 'youtuber_idx', 'channel_name',
        'final_score', 'baseline_corrected', 'raw_prob',
        'existing_collab_degree', 'degree_penalty', 'ensemble_std', 'candidate_rank_pct'
    ]
    cols = [c for c in cols if c in rec.columns]
    display(rec[cols])

    if export_csv:
        safe_brand = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in brand_name)[:80]
        path = os.path.join(output_dir, f'top{top_k}_{safe_brand}_recommendation.csv')
        rec[cols].to_csv(path, index=False, encoding='utf-8-sig')
        print(f'Saved: {path}')

    return rec, score_df


def recommend_many(brand_list, **kwargs):
    all_results = {}
    for b in brand_list:
        rec, score_df = recommend_brand(b, **kwargs)
        all_results[b] = {'recommendation': rec, 'all_scores': score_df}
    return all_results


## 7. Run recommendation

In [ ]:
# Edit the brand name and recommendation settings here
BRAND_LIST = ['Samsung']  
TOP_K = 5
SCORE_TYPE = 'final_score' 
POPULARITY_ALPHA = 0.08    

results = recommend_many(
    BRAND_LIST,
    top_k=TOP_K,
    score_type=SCORE_TYPE,
    popularity_alpha=POPULARITY_ALPHA,
    export_csv=True,
)
